### GLC-FCS30

In [ ]:
data_url = 'https://s3.openlandmap.org/arco/lc_glc.fcs30d_c_30m_s_20210101_20211231_go_epsg.4326_v20231026.tif'
glc_fcs_ds = rxr.open_rasterio(
    data_url, 
    chunks={'x': 1024, 'y': 1024},  # Explicitly define chunk sizes
)
glc_fcs_ds

In [ ]:
glc_fcs_da = glc_fcs_ds.squeeze()
glc_fcs_da

In [ ]:
bbox = aoi_gdf.geometry.total_bounds
glc_fcs_da_subset = glc_fcs_da.rio.clip_box(*bbox)
glc_fcs_da_subset

In [ ]:
%%time
glc_fcs_da_subset = glc_fcs_da_subset.compute()

In [ ]:
aoi = pyproj.aoi.AreaOfInterest(*bbox)
utm_crs_list = pyproj.database.query_utm_crs_info(datum_name='WGS 84', area_of_interest=aoi)
utm_crs = pyproj.CRS.from_epsg(utm_crs_list[0].code)
utm_crs

In [ ]:
glc_fcs_da_clipped = glc_fcs_da_subset.astype('float32').rio.clip(aoi_gdf.geometry)
glc_fcs_da_reprojected = glc_fcs_da_clipped.rio.reproject(utm_crs)
glc_fcs_da_reprojected

In [ ]:
glc_fcs_class_dict = {
    10: {'description': 'Rainfed cropland', 'hex': 'ffff64'},
    11: {'description': 'Herbaceous cover cropland', 'hex': 'ffff64'},
    12: {'description': 'Tree/shrub cover (Orchard) cropland', 'hex': 'ffff00'},
    20: {'description': 'Irrigated cropland', 'hex': 'aaf0f0'},
    51: {'description': 'Open evergreen broadleaved forest', 'hex': '4c7300'},
    52: {'description': 'Closed evergreen broadleaved forest', 'hex': '006400'},
    61: {'description': 'Open deciduous broadleaved forest', 'hex': 'aac800'},
    62: {'description': 'Closed deciduous broadleaved forest', 'hex': '00a000'},
    71: {'description': 'Open evergreen needle-leaved forest', 'hex': '005000'},
    72: {'description': 'Closed evergreen needle-leaved forest', 'hex': '003c00'},
    81: {'description': 'Open deciduous needle-leaved forest', 'hex': '286400'},
    82: {'description': 'Closed deciduous needle-leaved forest', 'hex': '285000'},
    91: {'description': 'Open mixed leaf forest', 'hex': 'a0b432'},
    92: {'description': 'Closed mixed leaf forest', 'hex': '788200'},
    120: {'description': 'Shrubland', 'hex': '966400'},
    121: {'description': 'Evergreen shrubland', 'hex': '964b00'},
    122: {'description': 'Deciduous shrubland', 'hex': '966400'},
    130: {'description': 'Grassland', 'hex': 'ffb432'},
    140: {'description': 'Lichens and mosses', 'hex': 'ffdcd2'},
    150: {'description': 'Sparse vegetation', 'hex': 'ffebaf'},
    152: {'description': 'Sparse shrubland', 'hex': 'ffd278'},
    153: {'description': 'Sparse herbaceous', 'hex': 'ffebaf'},
    181: {'description': 'Swamp', 'hex': '00a884'},
    182: {'description': 'Marsh', 'hex': '73ffdf'},
    183: {'description': 'Flooded flat', 'hex': '9ebbd7'},
    184: {'description': 'Saline', 'hex': '828282'},
    185: {'description': 'Mangrove', 'hex': 'f57ab6'},
    186: {'description': 'Salt marsh', 'hex': '66cdab'},
    187: {'description': 'Tidal flat', 'hex': '444f89'},
    190: {'description': 'Impervious surfaces', 'hex': 'c31400'},
    200: {'description': 'Bare areas', 'hex': 'fff5d7'},
    201: {'description': 'Consolidated bare areas', 'hex': 'dcdcdc'},
    202: {'description': 'Unconsolidated bare areas', 'hex': 'fff5d7'},
    210: {'description': 'Water body', 'hex': '0046c8'},
    220: {'description': 'Permanent ice and snow', 'hex': 'ffffff'},
}

colors = ['#000000'] * 256
for key, value in glc_fcs_class_dict.items():
    colors[key] = f'#{value["hex"]}'
colors[0] = (0, 0, 0, 0)  # transparent for nodata
glc_cmap = matplotlib.colors.ListedColormap(colors)
normalizer = matplotlib.colors.Normalize(vmin=0, vmax=255)

unique_classes = sorted(glc_fcs_class_dict.keys())
boundaries = [(unique_classes[i + 1] + unique_classes[i]) / 2
              for i in range(len(unique_classes) - 1)]
boundaries = [0] + boundaries + [255]
ticks = [(boundaries[i + 1] + boundaries[i]) / 2
         for i in range(len(boundaries) - 1)]
tick_labels = [f'{v["description"]} ({k})' for k, v in glc_fcs_class_dict.items()]

fig, ax = plt.subplots(1, 1)
fig.set_size_inches(12, 14)

glc_fcs_da_clipped.plot(ax=ax, cmap=glc_cmap, norm=normalizer)

colorbar = fig.colorbar(
    cm.ScalarMappable(norm=normalizer, cmap=glc_cmap),
    boundaries=boundaries,
    values=unique_classes,
    cax=fig.axes[1].axes,
)
colorbar.set_ticks(ticks, labels=tick_labels)

ax.set_axis_off()
ax.set_title('Landcover Classes from GLC_FCS30D')